In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


d:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.chdir("D:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot")


<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
C:\Users\ashvi\AppData\Local\Temp\ipykernel_25772\4218916916.py:2: SyntaxWarning: invalid escape sequence '\M'
  os.chdir("D:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot")


In [3]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "./Qwen2.5-1.5B"
SYSTEM_PROMPT = "You are a helpful customer support assistant."

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("Chatbot ready. Type 'exit' to quit.\n")

history = [{"role": "system", "content": SYSTEM_PROMPT}]
while True:
    user_input = input("You: ").strip()
    if user_input.lower() == "exit":
        break

    history.append({"role": "user", "content": user_input})

    prompt = tokenizer.apply_chat_template(
        history, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    print(f"Bot: {response}\n")
    history.append({"role": "assistant", "content": response})


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W0623 14:16:23.859000 25772 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 1/338 [00:00<03:07,  1.80it/s]d:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 338/338 [00:04<00:00, 81.27it/s]


Chatbot ready. Type 'exit' to quit.



d:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Bot: Thank you for reaching out! I'm here to help you with entering the information needed. Could you please provide more details about what specific information you need to enter? This will allow me to assist you better and ensure that your entry is accurate and relevant.

Bot: I see what you mean! You're looking to enter some information, and I'm here to guide you through the process. To get started, could you please let me know which system or platform you want to enter the information into? Once I have this information, I'll be able to provide you with step-by-step instructions on how to proceed.

Bot: I've understood, you would like to cancel the order you made yesterday. We apologize for any inconvenience caused by this situation. Our team is dedicated to assisting you in resolving this matter promptly.

To initiate the cancellation process, please follow these steps:

1. Log in to your {{Online Company Portal Info}} using your credentials.
2. Navigate to the '{{Online Order Inte

In [4]:
training_params = {
    "num_train_epochs" : 2,                     # 2 epochs is plenty on 5000 diverse examples; # 3+ risks overfitting/memorizing responses
    "per_device_train_batch_size" : 1,          # do not raise this on 4GB
    "per_device_eval_batch_size" : 1,
    "gradient_accumulation_steps" : 8,          # effective batch size = 1 * 8 = 8
    "gradient_checkpointing" : "True",
    "optim" : "paged_adamw_8bit",               # paged optimizer avoids OOM spikes
    "learning_rate" : "2e-4",
    "lr_scheduler_type" : "cosine",
    "warmup_ratio" : 0.03,
    "logging_steps" : 10,
    "eval_strategy" : "steps",
    "eval_steps" : 100,
    "save_strategy" : "steps",
    "save_steps" : 100,
    "save_total_limit" : 3,
    "bf16":"True",                              # [if no bf16 support] set fp16=True instead
    "max_length" : 512,
    "dataset_text_field" : "text",
    "packing" : "False",                          # keep False for clearer per-example loss
    "report_to" : "none"
}


In [ ]:
metrics = {
    "training loss" : 0.548204, 
    "evaluation loss" : 0.617285,
    "Entropy" : 0.562029,
    "Accuracy" : 0.810830
}


In [6]:
import mlflow


In [7]:
# Merge adapter into base model
merged_model = model.merge_and_unload()

# Save merged model
merged_model.save_pretrained("./merged_model")
tokenizer.save_pretrained("./merged_model")


d:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot\venv\Lib\site-packages\peft\tuners\lora\bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
d:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.40s/it]


('./merged_model\\tokenizer_config.json',
 './merged_model\\chat_template.jinja',
 './merged_model\\tokenizer.json')

In [12]:
mlflow.set_experiment("Customer-Support-Chat-Bot")
mlflow.set_tracking_uri("http://127.0.0.1:5000/")

with mlflow.start_run(run_name="first-cs-chat-model"):
    mlflow.log_params(training_params)
    
    mlflow.log_metrics({
        "Training Loss" : metrics["training loss"],
        "Evaluation Loss" : metrics["evaluation loss"],
        "Entropy" : metrics["Entropy"],
        "Accuracy" : metrics["Accuracy"]
    })
    
    mlflow.transformers.log_model(
        transformers_model={
            "model" : merged_model,
            "tokenizer" : tokenizer
        },
        name = "Customer-Support-ChatBot-Qwen2_5-1_5B-fine-tuned-model"
    )


Writing model shards: 100%|██████████| 3/3 [00:02<00:00,  1.41it/s]
d:\My Projects\LLM Fine Tuning\Customer-Support-Chat-Bot\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ashvi\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
2026/06/23 1

🏃 View run first-cs-chat-model at: http://127.0.0.1:5000/#/experiments/1/runs/93c18152a5bd47f6a37f3ca3df41b1e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [13]:
model_name = "Customer-Support-ChatBot-Qwen2_5-1_5B-fine-tuned-model"
run_id = "93c18152a5bd47f6a37f3ca3df41b1e2"
model_uri = f"runs:/{run_id}/{model_name}"
result = mlflow.register_model(
    model_uri, model_name
)


Successfully registered model 'Customer-Support-ChatBot-Qwen2_5-1_5B-fine-tuned-model'.
2026/06/23 15:42:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 93c18152a5bd47f6a37f3ca3df41b1e2 has no artifacts at artifact path 'Customer-Support-ChatBot-Qwen2_5-1_5B-fine-tuned-model', registering model based on models:/m-7679083c9cfc453c9c3c9dad6c17f491 instead
2026/06/23 15:42:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Customer-Support-ChatBot-Qwen2_5-1_5B-fine-tuned-model, version 1
Created version '1' of model 'Customer-Support-ChatBot-Qwen2_5-1_5B-fine-tuned-model'.


In [15]:
# DAGSHUB

import dagshub
dagshub.init(repo_owner="iam.ashvindhakal", repo_name="Customer-Support-ChatBot" , mlflow=True)


Accessing as iam.ashvindhakal

Initialized MLflow to track repo "iam.ashvindhakal/Customer-Support-ChatBot"

Repository iam.ashvindhakal/Customer-Support-ChatBot initialized!

In [16]:
mlflow.set_experiment("Customer-Support-Chat-Bot")
mlflow.set_tracking_uri("https://dagshub.com/iam.ashvindhakal/Customer-Support-ChatBot.mlflow")

with mlflow.start_run(run_name="first-cs-chat-model"):
    mlflow.log_params(training_params)
    
    mlflow.log_metrics({
        "Training Loss" : metrics["training loss"],
        "Evaluation Loss" : metrics["evaluation loss"],
        "Entropy" : metrics["Entropy"],
        "Accuracy" : metrics["Accuracy"]
    })
    
    mlflow.transformers.log_model(
        transformers_model={
            "model" : merged_model,
            "tokenizer" : tokenizer
        },
        name = "Customer-Support-ChatBot-Qwen2_5-1_5B-fine-tuned-model"
    )


2026/06/23 15:50:12 INFO mlflow.tracking.fluent: Experiment with name 'Customer-Support-Chat-Bot' does not exist. Creating a new experiment.
Writing model shards: 100%|██████████| 3/3 [00:02<00:00,  1.21it/s]
2026/06/23 15:50:22 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/23 15:50:22 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/23 15:50:22 WAR

🏃 View run first-cs-chat-model at: https://dagshub.com/iam.ashvindhakal/Customer-Support-ChatBot.mlflow/#/experiments/0/runs/d7887eade7154706b9c9e0eed1e5ba18
🧪 View experiment at: https://dagshub.com/iam.ashvindhakal/Customer-Support-ChatBot.mlflow/#/experiments/0
